In [1]:
from openpyxl import load_workbook
import sys

wb = load_workbook("./生産計画自動計算.xlsx")   # ブックを開く
ws = wb["処理対象"]   # シートを取り出す
y = ws["A1"].value    # 対象の年をyへ代入
m = ws["A2"].value    # 対象の月をmへ代入
target_ws = wb[f"{y}.{m}"]   # 対象のワークシートをtarget_wsへ代入
output_ws_name = f"{y}.{m}" + "_出力"  # 出力するワークシート名
order_cnt = {} # 発注回数の辞書

# 対象のワークシートの一か月前のワークシートをpre_target_wsへ代入
if m == 1:
    pre_target_ws = f"{y-1}.{12}"
else:
    pre_target_ws = f"{y}.{m-1}"
if pre_target_ws not in wb.sheetnames:
    pre_flag = 0
    if (not isinstance(target_ws["B12"].value, int)
            or not isinstance(target_ws["B13"].value, int)
            or not isinstance(target_ws["B14"].value, int)):
        print("前月末在庫（B12〜B14）を入力してください")
        sys.exit(1)
else:
    pre_flag = 1
    pre_target_ws = pre_target_ws + "_出力"
    if pre_target_ws not in wb.sheetnames:
        print(f"{pre_target_ws}がありません。前月を先に実行してください。")
        sys.exit(1)
    else:
        pre_target_ws = wb[pre_target_ws]


# 日付が列3～列Xまで存在する。最後の列番号をlast_colへ代入して戻り値とする
def last_col(ws):
    col = 3   # C列から始める
    while ws.cell(row=2, column=col).value is not None:
        col += 1
    last_col = col - 1   # 空欄の1つ手前が最後の列
    #print(last_col)  # A〜AF なら 30、A〜AG なら 31
    return last_col


# 部品クラスの宣言
class Part:
    def __init__(self, kind):
        self.kind = kind
        self.order_cnt = 0
        self.threshold, self.order_qty = self.input_default()
        self.premonth_stock = self.premonth_stock()
        self.stock_dict = {}
        self.prod_dict = {}
        self.last_column = last_col(target_ws)

    # 閾値と発注量を変数に代入
    def input_default(self):
        if self.kind == "A":
            threshold = target_ws["B17"].value
            order_qty = target_ws["C17"].value
        elif self.kind == "B":
            threshold = target_ws["B18"].value
            order_qty = target_ws["C18"].value
        elif self.kind == "C":
            threshold = target_ws["B19"].value
            order_qty = target_ws["C19"].value
        else:
            print("Partクラスの引数の指定が異常です。")
            sys.exit(1)
        return threshold, order_qty


    # 前月末在庫量をpremonth_stockへ代入し、戻り値とする
    def premonth_stock(self):
        B12 = target_ws["B12"].value
        B13 = target_ws["B13"].value
        B14 = target_ws["B14"].value

        if pre_flag == 1:
            pre_last_col = last_col(pre_target_ws)
            pre_B12 = pre_target_ws.cell(row=12, column=pre_last_col).value
            pre_B13 = pre_target_ws.cell(row=13, column=pre_last_col).value
            pre_B14 = pre_target_ws.cell(row=14, column=pre_last_col).value
            if self.kind == "A":
                premonth_stock = pre_B12
            if self.kind == "B":
                premonth_stock = pre_B13
            if self.kind == "C":
                premonth_stock = pre_B14

        elif pre_flag == 0:
            if self.kind == "A":
                premonth_stock = B12
            if self.kind == "B":
                premonth_stock = B13
            if self.kind == "C":
                premonth_stock = B14
        return premonth_stock


    # 生産量と在庫量を計算してprod_dictとstock_dictに代入する。
    def count(self):
        for col in range(3, self.last_column + 1):
            if self.kind == "A":
                cons = target_ws.cell(row=4, column=col).value
            elif self.kind == "B":
                cons = target_ws.cell(row=5, column=col).value
            elif self.kind == "C":
                cons = target_ws.cell(row=6, column=col).value

            if col == 3:
                preday_stock = self.premonth_stock
            else:
                preday_stock = self.stock_dict[col-1]

            stock = preday_stock - cons
            if target_ws.cell(row=2, column=col).value.weekday() > 4:
                prod = 0
            else:
                if preday_stock <= self.threshold:
                    prod = self.order_qty
                    stock = stock + prod
                    self.order_cnt += 1
                else:
                    prod = 0
            self.prod_dict[col]= prod
            self.stock_dict[col] = stock


    def write_ws(self, ws_out):
        for col in range(3, self.last_column + 1):
            if self.kind == "A":
                ws_out.cell(row=8, column=col).value = self.prod_dict[col]
                ws_out.cell(row=12, column=col).value = self.stock_dict[col]
            elif self.kind == "B":
                ws_out.cell(row=9, column=col).value = self.prod_dict[col]
                ws_out.cell(row=13, column=col).value = self.stock_dict[col]
            elif self.kind == "C":
                ws_out.cell(row=10, column=col).value = self.prod_dict[col]
                ws_out.cell(row=14, column=col).value = self.stock_dict[col]

        if pre_flag == 1:
            if self.kind == "A":
                ws_out.cell(row = 12, column = 2).value = self.premonth_stock
            elif self.kind == "B":
                ws_out.cell(row = 13, column = 2).value = self.premonth_stock
            elif self.kind == "C":
                ws_out.cell(row = 14, column = 2).value = self.premonth_stock

def set_pos_ws_out(ws_out):
    target_pos = wb.sheetnames.index(target_ws.title)
    output_pos = wb.sheetnames.index(output_ws_name)
    offset = output_pos - target_pos
    move = -(offset - 1)
    wb.move_sheet(ws_out, move)


def main():
    if output_ws_name in wb.sheetnames:
        wb.remove(wb[output_ws_name])
    ws_out = wb.copy_worksheet(target_ws)
    ws_out.title = output_ws_name
    set_pos_ws_out(ws_out)

    string = ["A", "B", "C"]
    for kind in string:
        p = Part(kind)
        p.count()
        p.write_ws(ws_out)
        order_cnt[kind] = p.order_cnt

    try:
        wb.save("生産計画自動計算.xlsx")
        print(f"{output_ws_name}を作成しました。")
        for kind in string:
            print(f"部品{kind}の発注：{order_cnt[kind]}回")
    except PermissionError:
        print("保存できません。Excelでファイルを閉じてから実行してください。")
        sys.exit(1)

# このファイルを直接実行したときだけ main() を呼び出す、というお決まりの書き方
if __name__ == "__main__":
    main()


2026.10_出力を作成しました。
部品Aの発注：7回
部品Bの発注：8回
部品Cの発注：12回
